In [1]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import _pickle as pickle

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = '/home/trade_core' # TEST
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.upbit_plug import UserUpbitAdaptor
from exchange_plugin.binance_plug import UserBinanceAdaptor
from etc.redis_connector.redis_connector import InitRedis

In [2]:
binance_access_key = '4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb'
binance_secret_key = '011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK'

upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

okx_access_key = "fcb66a6e-0443-4743-8d9b-61caf7eab662"
okx_secret_key = "0029E09874B38F5AC7E68E9DFC348667"
okx_passphrase = "121431Rn!!"

bybit_access_key = "S3YKBbD0kz1WwcfqZD"
bybit_secret_key = "390M6dKJ9J0uEK7vXYAl3qxVCh944tRrzHah"

In [36]:
class UserExchangeAdaptor:
    def __init__(self, logging_dir=None):
        self.logger = KimpBotLogger("integrated_plug", logging_dir=logging_dir).logger
        self.logger.info('integrated_plug_logger init')
        self.exchange_adaptor_dict = {}
        self.exchange_adaptor_dict["UPBIT"] = UserUpbitAdaptor(logging_dir=logging_dir)
        self.exchange_adaptor_dict["BINANCE"] = UserBinanceAdaptor(logging_dir=logging_dir)
        # redis connection to read info_dict
        self.local_redis_client = InitRedis(host='localhost', port=6379, db=0, passwd=None)

    def check_api_key(self, exchange, access_key, secret_key, futures=False):
        """FUTURES includes USD_M, COIN_M"""
        if exchange.upper() not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        return self.exchange_adaptor_dict[exchange].check_api_key(access_key, secret_key, futures)
    
    def get_position(self, exchange, access_key, secret_key, market_type):
        """SPOT position columns: ['asset', 'free', 'locked']
        USD_M position columns: []
        """

        if exchange.upper() not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        exchange_adaptor = self.exchange_adaptor_dict[exchange]
        if exchange == "UPBIT":
            position_df = exchange_adaptor.get_balance(access_key, secret_key, market_type)
        elif exchange == "BINANCE":
            position_df = exchange_adaptor.all_position_information(access_key, secret_key, market_type)
            info_df = pickle.loads(self.local_redis_client.get_data(f'TRADE_CORE|{exchange.lower()}_{market_type.lower()}_info_df'))
            position_df = position_df.merge(info_df[['symbol','base_asset']], how='left', on='symbol')
            position_df = position_df.rename(columns={"positionAmt":"qty", "marginType":"margin_type", "entryPrice":"entry_price", "liquidationPrice":"liquidation_price"})
            position_df["ROI"] = position_df.apply(lambda x: (x['entry_price']-x['markPrice'])/x['markPrice']*x['leverage']*100 if x['qty']<0 else 
                                                   (x['markPrice']-x['entry_price'])/x['entry_price']*['leverage']*100, axis=1)
        else:
            raise Exception(f'exchange {exchange} not supported')
        return position_df

In [37]:
user_exchange_adaptor = UserExchangeAdaptor()

2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plu

In [11]:
pickle.loads(user_exchange_adaptor.local_redis_client.get_data('TRADE_CORE|binance_usd_m_info_df'))

,symbol,pair,contractType,deliveryDate,onboardDate,status,maintMarginPercent,requiredMarginPercent,base_asset,quote_asset,...,underlyingSubType,settlePlan,triggerProtect,liquidationFee,marketTakeBound,maxMoveOrderLimit,filters,orderTypes,timeInForce,perpetual
0,BTCUSDT,BTCUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,BTC,USDT,...,[PoW],0,0.05,0.0125,0.05,10000,"[{'maxPrice': '4529764', 'tickSize': '0.10', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
1,ETHUSDT,ETHUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,ETH,USDT,...,[Layer-1],0,0.05,0.0125,0.05,10000,"[{'minPrice': '39.86', 'tickSize': '0.01', 'ma...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
2,BCHUSDT,BCHUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,BCH,USDT,...,[PoW],0,0.05,0.0150,0.05,10000,"[{'tickSize': '0.01', 'minPrice': '13.93', 'ma...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
3,XRPUSDT,XRPUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,XRP,USDT,...,[Payment],0,0.05,0.0125,0.05,10000,"[{'maxPrice': '100000', 'minPrice': '0.0143', ...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
4,EOSUSDT,EOSUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,EOS,USDT,...,[Layer-1],0,0.10,0.0100,0.10,10000,"[{'minPrice': '0.111', 'maxPrice': '100000', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,RONINUSDT,RONINUSDT,PERPETUAL,4133404800000,1707193800000,TRADING,2.5,5.0,RONIN,USDT,...,[Layer-1],0,0.15,0.0150,0.15,10000,"[{'tickSize': '0.001000', 'minPrice': '0.00100...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
274,DYMUSDT,DYMUSDT,PERPETUAL,4133404800000,1707289200000,TRADING,2.5,5.0,DYM,USDT,...,[Infrastructure],0,0.15,0.0150,0.15,10000,"[{'minPrice': '0.001000', 'filterType': 'PRICE...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
275,SUIUSDC,SUIUSDC,PERPETUAL,4133404800000,1707375600000,TRADING,2.5,5.0,SUI,USDC,...,[USDC],0,0.15,0.0150,0.15,10000,"[{'filterType': 'PRICE_FILTER', 'minPrice': '0...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
276,OMUSDT,OMUSDT,PERPETUAL,4133404800000,1707832800000,TRADING,2.5,5.0,OM,USDT,...,[DeFi],0,0.15,0.0150,0.15,10000,"[{'filterType': 'PRICE_FILTER', 'tickSize': '0...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True


In [40]:
user_exchange_adaptor.get_position('BINANCE', binance_access_key, binance_secret_key, 'USD_M')[['base_asset','qty','liquidation_price','ROI']]

,base_asset,qty,liquidation_price,ROI
0,ETC,-291.78,122.477761,-23.569741
1,STX,-4944.0,8.104302,-167.206288
2,SOL,-42.0,779.346108,-91.794215
3,GMT,-15175.0,2.110701,-51.157663
4,XRP,-10227.0,3.297133,6.933911
5,SEI,-18497.0,2.410834,-38.883366
6,LSK,-2442.0,12.62177,31.881503
7,VET,-151592.0,0.228765,25.057258
